# Chicago Weather Fetcher Playground

Use this notebook to explore the NOAA weather fetcher components in `core.weather_fetcher`.

In [4]:
from core import weather_fetcher, geo_utils, config
import pandas as pd

# Rough Chicago viewport (you can tweak these)
CHICAGO_CENTER = {"lat": 41.8781, "lon": -87.6298}
CHICAGO_BBOX = geo_utils.BoundingBox(
    lat_min=41.60,
    lat_max=42.10,
    lon_min=-88.00,
    lon_max=-87.40,
 )

print("Weather API:", config.WEATHER_BASE_URL)
print("Station limit:", config.WEATHER_STATION_LIMIT)
print("Point sample limit:", config.WEATHER_POINT_SAMPLE_LIMIT)

Weather API: https://api.weather.gov
Station limit: 10
Point sample limit: 5


In [5]:
# 1) Pure geometry component: sampled points used to discover nearby NOAA station lists
sample_points = weather_fetcher._sample_points(CHICAGO_BBOX, CHICAGO_CENTER)
sample_points

[(41.8781, -87.6298),
 (41.6, -88.0),
 (41.6, -87.4),
 (42.1, -88.0),
 (42.1, -87.4)]

In [6]:
# 2) Network component: candidate stations near sampled points
candidates = weather_fetcher._station_candidates(CHICAGO_BBOX, CHICAGO_CENTER)
candidates_df = pd.DataFrame(candidates)
candidates_df.head(10)

,station_id,name,latitude,longitude,distance_to_center_km
0,KMDW,"Chicago, Chicago Midway Airport",41.78417,-87.75528,14.736789
1,KORD,"Chicago, Chicago-O'Hare International Airport",41.97972,-87.90444,25.374701
2,KGYY,Gary Regional Airport,41.61212,-87.40908,34.785614
3,KPWK,"Chicago / Wheeling, Pal-Waukee Airport",42.12083,-87.90472,35.278593
4,KIGQ,Lansing Municipal Airport,41.54125,-87.52822,38.393399
5,KLOT,Lewis University Airport,41.60307,-88.10164,49.677459
6,KDPA,"Chicago / West Chicago, Dupage Airport",41.89639,-88.25111,51.472245
7,KJOT,Joliet Regional Airport,41.51755,-88.17903,60.717687
8,KUGN,Chicago/Waukegan Regional Airport,42.42546,-87.86339,63.837149
9,KVPZ,Valparaiso Porter County Municipal Airport,41.45349,-86.99805,70.590484


In [7]:
# 3) End-to-end component: fetch + normalize latest observations
overlay = weather_fetcher.fetch_workspace_weather(CHICAGO_BBOX, CHICAGO_CENTER)

print("Available:", overlay.get("available"))
print("Summary:")
for line in overlay.get("summary", []):
    print(" -", line)

weather_df = pd.DataFrame(overlay.get("points", []))
weather_df.head(10)

Available: True
Summary:
 - Weather stations loaded: 10
 - Request failures: 0
 - Latest observation: 2026-03-15T18:00:00+00:00
 - Cloudy / obstructed stations: 10
 - Mean wind speed: 48.7 mph
 - Mean temperature: 55.5 °F


,station_id,name,latitude,longitude,distance_to_center_km,timestamp,temperature_c,temperature_f,wind_speed_m_s,wind_speed_mph,wind_direction_deg,wind_direction_cardinal,pressure_pa,pressure_hpa,cloud_amounts,has_clouds,text_description
0,KMDW,"Chicago, Chicago Midway Airport",41.78417,-87.75528,14.736789,2026-03-15T18:00:00+00:00,16.0,60.80,29.628,66.276058,190.0,S,99390.51,993.9051,[OVC],True,Cloudy
1,KORD,"Chicago, Chicago-O'Hare International Airport",41.97972,-87.90444,25.374701,2026-03-15T18:00:00+00:00,14.0,57.20,NaN,NaN,NaN,NaN,99322.78,993.2278,"[FEW, OVC]",True,Cloudy
2,KGYY,Gary Regional Airport,41.61212,-87.40908,34.785614,2026-03-15T17:50:00+00:00,15.0,59.00,24.120,53.954993,200.0,S,99460.00,994.6000,[OVC],True,Light Rain
3,KPWK,"Chicago / Wheeling, Pal-Waukee Airport",42.12083,-87.90472,35.278593,2026-03-15T18:00:00+00:00,14.0,57.20,NaN,NaN,NaN,NaN,99356.64,993.5664,"[SCT, BKN, OVC]",True,Cloudy
4,KIGQ,Lansing Municipal Airport,41.54125,-87.52822,38.393399,2026-03-13T15:50:00+00:00,4.1,39.38,NaN,NaN,NaN,NaN,100200.00,1002.0000,[OVC],True,Cloudy
5,KLOT,Lewis University Airport,41.60307,-88.10164,49.677459,2026-03-15T17:45:00+00:00,14.0,57.20,NaN,NaN,NaN,NaN,99430.00,994.3000,[OVC],True,Cloudy
6,KDPA,"Chicago / West Chicago, Dupage Airport",41.89639,-88.25111,51.472245,2026-03-15T18:00:00+00:00,13.0,55.40,NaN,NaN,NaN,NaN,99288.91,992.8891,"[SCT, OVC]",True,Light Rain
7,KJOT,Joliet Regional Airport,41.51755,-88.17903,60.717687,2026-03-15T17:55:00+00:00,15.7,60.26,NaN,NaN,NaN,NaN,99430.00,994.3000,"[SCT, OVC]",True,Haze
8,KUGN,Chicago/Waukegan Regional Airport,42.42546,-87.86339,63.837149,2026-03-15T18:00:00+00:00,9.0,48.20,5.544,12.401595,250.0,W,99288.91,992.8891,"[FEW, BKN, OVC]",True,Cloudy
9,KVPZ,Valparaiso Porter County Municipal Airport,41.45349,-86.99805,70.590484,2026-03-15T18:00:00+00:00,16.0,60.80,27.792,62.169036,190.0,S,99627.55,996.2755,[OVC],True,Cloudy


In [8]:
# 4) Playground helper: quickly rerun with custom bounding boxes
def run_weather_playground(lat_min=41.60, lat_max=42.10, lon_min=-88.00, lon_max=-87.40, center_lat=41.8781, center_lon=-87.6298):
    bbox = geo_utils.BoundingBox(lat_min=lat_min, lat_max=lat_max, lon_min=lon_min, lon_max=lon_max)
    center = {"lat": center_lat, "lon": center_lon}
    data = weather_fetcher.fetch_workspace_weather(bbox, center)
    df = pd.DataFrame(data.get("points", []))
    return data.get("summary", []), df

summary, df = run_weather_playground()
summary, df.head(5)

(['Weather stations loaded: 10',
  'Request failures: 0',
  'Latest observation: 2026-03-15T18:00:00+00:00',
  'Cloudy / obstructed stations: 10',
  'Mean wind speed: 48.7 mph',
  'Mean temperature: 55.5 °F'],
   station_id                                           name  latitude  \
 0       KMDW                Chicago, Chicago Midway Airport  41.78417   
 1       KORD  Chicago, Chicago-O'Hare International Airport  41.97972   
 2       KGYY                          Gary Regional Airport  41.61212   
 3       KPWK         Chicago / Wheeling, Pal-Waukee Airport  42.12083   
 4       KIGQ                      Lansing Municipal Airport  41.54125   
 
    longitude  distance_to_center_km                  timestamp  temperature_c  \
 0  -87.75528              14.736789  2026-03-15T18:00:00+00:00           16.0   
 1  -87.90444              25.374701  2026-03-15T18:00:00+00:00           14.0   
 2  -87.40908              34.785614  2026-03-15T17:50:00+00:00           15.0   
 3  -87.90472   